# KLEOS 01 — Dataset validationValidate a dataset before training on it. Everything here runs on CPU in secondsand needs no GPU, so you can do dataset work on any runtime.What gets checked:- **schema** — every example matches the versioned data contract- **duplicates and leakage** — exact, normalized and near-duplicates, id  collisions, scenario repeats, entity leakage- **coverage** — how many examples cover each *situation type*, not just how many  examples there are- **privacy** — patterns that suggest private data reached a public repository- **policy vs facts** — examples that teach a private fact instead of a decision  policy

## 1. Setup

In [ ]:
# Clone the repository (skip if already present) and enter it.import osfrom pathlib import PathREPO_DIR = Path("/content/kleos-models")if not REPO_DIR.exists():    !git clone https://github.com/kleos/kleos-models.git {REPO_DIR}else:    print(f"{REPO_DIR} already exists; pulling latest")    !cd {REPO_DIR} && git pull --ff-onlyos.chdir(REPO_DIR)print("working directory:", Path.cwd())

In [ ]:
# Install dependencies WITHOUT touching Colab's torch build.## Reinstalling torch on Colab replaces the build compiled against this runtime's# CUDA driver, and CUDA then silently stops working. scripts/colab_setup.py uses# --no-deps for every package that would otherwise pull torch along.!python scripts/colab_setup.py

## 2. Choose the datasetBy default this validates the bundled **development fixtures**, which aresynthetic and exist only to exercise the pipeline. They are not the KLEOSresearch dataset.To validate the real dataset produced by the private `kleos-training-data`repository, mount Drive and point `DATASET` at it.

In [ ]:
DATASET = "data/examples"   # or "/content/drive/MyDrive/kleos/dataset"# Uncomment to mount Drive for a private dataset:# from google.colab import drive# drive.mount("/content/drive")print("validating:", DATASET)

## 3. Validate schema, content and leakage

In [ ]:
!python scripts/validate_dataset.py --dataset {DATASET} --leakage-report reports/leakage

A cross-split duplicate is the finding that matters most: it means an evaluationexample is effectively present in training, and any "generalization" numbercomputed from it is measuring memorization.

## 4. Quality and coverage report

In [ ]:
!python scripts/inspect_dataset.py --dataset {DATASET} --coverage

### What to look for- **Constant axes.** An axis with one value cannot support a claim about that  axis.- **Empty cells.** Situation types with no examples at all.- **Thin cells.** Present but too sparse to support a per-cell claim.- **Missing metadata.** Especially `scenario_family`, without which consistency  testing cannot group anything.A dataset is not diverse because it is large.

## 5. Exact token statistics (optional)Approximate counts are fine for triage. For the real numbers, load the tokenizerof the model you plan to train.

In [ ]:
!python scripts/inspect_dataset.py --dataset {DATASET} --tokenizer Qwen/Qwen3-8B

## 6. Create a versioned splitRandom splitting is development-only. Generalization claims need a held-outstrategy — otherwise paraphrases of the same scenario land on both sides of theboundary.

In [ ]:
# Entity-held-out: does the learned policy transfer to unseen entities?!python scripts/split_dataset.py \    --input {DATASET}/synthetic_train.jsonl \    --output /content/kleos-dataset-v1 \    --strategy entity_holdout \    --seed 42 \    --version kleos-policy-v0.1.0

The split is verified for overlap before it is written, and the manifest recordsthe strategy, seed, held-out values and per-file hashes.## Next`02_train_qlora.ipynb` — train an adapter on this dataset.